### RegulonDB Regulon/Sigmulon prediction evaluation

In [ ]:
from promoter_protein_clip.model_transformer import SequenceCLIPTransformer
import torch
from huggingface_hub import hf_hub_download
import os
from promoter_protein_clip.utils import get_project_root
from promoter_protein_clip.evaluation import *
from sklearn.metrics import pairwise_distances
from transformers import GPT2LMHeadModel, AutoModel, AutoTokenizer
import pickle

### Loading models

In [ ]:
# Loading trained C3P models
repo_id = "dufaultc/contrastive-promoter-protein-pretraining"

filename = "C3P_100M.pth"
checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename )
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")    
checkpoint= torch.load(checkpoint_path, map_location=device, weights_only=False)    
model_100 = SequenceCLIPTransformer(
    protein_model_name=checkpoint['args'].protein_model,
    promoter_embedding_dim=checkpoint['args'].promoter_embedding_dim,
    depth=checkpoint['args'].transformer_depth,
    num_heads=checkpoint['args'].transformer_num_heads,
    max_promoter_length=checkpoint['args'].max_promoter_length,
    projection_dim=checkpoint['args'].projection_dim,       
 ).to(device)
model_100 = torch.compile(model_100)
model_100.load_state_dict(checkpoint['model_state_dict'])  


filename = "C3P_25M.pth"
checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename )
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")    
checkpoint= torch.load(checkpoint_path, map_location=device, weights_only=False)    
model_25 = SequenceCLIPTransformer(
    protein_model_name=checkpoint['args'].protein_model,
    promoter_embedding_dim=checkpoint['args'].promoter_embedding_dim,
    depth=checkpoint['args'].transformer_depth,
    num_heads=checkpoint['args'].transformer_num_heads,
    max_promoter_length=checkpoint['args'].max_promoter_length,
    projection_dim=checkpoint['args'].projection_dim,       
 ).to(device)
model_25 = torch.compile(model_25)
model_25.load_state_dict(checkpoint['model_state_dict'])  

filename = "C3P_5M.pth"
checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename )
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")    
checkpoint= torch.load(checkpoint_path, map_location=device, weights_only=False)    
model_5 = SequenceCLIPTransformer(
    protein_model_name=checkpoint['args'].protein_model,
    promoter_embedding_dim=checkpoint['args'].promoter_embedding_dim,
    depth=checkpoint['args'].transformer_depth,
    num_heads=checkpoint['args'].transformer_num_heads,
    max_promoter_length=checkpoint['args'].max_promoter_length,
    projection_dim=checkpoint['args'].projection_dim,       
 ).to(device)
model_5 = torch.compile(model_5)
model_5.load_state_dict(checkpoint['model_state_dict']) 

filename = "C3P_1M.pth"
checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename )
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")    
checkpoint= torch.load(checkpoint_path, map_location=device, weights_only=False)    
model_1 = SequenceCLIPTransformer(
    protein_model_name=checkpoint['args'].protein_model,
    promoter_embedding_dim=checkpoint['args'].promoter_embedding_dim,
    depth=checkpoint['args'].transformer_depth,
    num_heads=checkpoint['args'].transformer_num_heads,
    max_promoter_length=checkpoint['args'].max_promoter_length,
    projection_dim=checkpoint['args'].projection_dim,       
 ).to(device)
model_1 = torch.compile(model_1)
model_1.load_state_dict(checkpoint['model_state_dict'])  

Loading weights: 100%|██████████| 486/486 [00:00<00:00, 18285.51it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 486/486 [00:00<00:00, 17726.73it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.

<All keys matched successfully>

In [ ]:
#Loading PromoGen2 models
llm_model = GPT2LMHeadModel.from_pretrained("jinyuan22/promogen2-base").to(device)
llm_model_small = GPT2LMHeadModel.from_pretrained("jinyuan22/promogen2-small").to(device)
llm_model_xsmall = GPT2LMHeadModel.from_pretrained("jinyuan22/promogen2-xsmall").to(device)
llm_tokenizer = AutoTokenizer.from_pretrained("jinyuan22/promogen2-base")

llm_model = torch.compile(llm_model)
llm_model.eval()

llm_model_small = torch.compile(llm_model_small)
llm_model_small.eval()

llm_model_xsmall = torch.compile(llm_model_xsmall)
llm_model_xsmall.eval()

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 13698.63it/s]


OptimizedModule(
  (_orig_mod): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(10, 320)
      (wpe): Embedding(256, 320)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-5): 6 x GPT2Block(
          (ln_1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D(nf=960, nx=320)
            (c_proj): Conv1D(nf=320, nx=320)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D(nf=1280, nx=320)
            (c_proj): Conv1D(nf=320, nx=1280)
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_features=320, out_features

In [21]:
#Loading gLM2 models
glm2_650_model = AutoModel.from_pretrained('tattabio/gLM2_650M', torch_dtype=torch.bfloat16, trust_remote_code=True).to(device)
glm2_150_model = AutoModel.from_pretrained('tattabio/gLM2_150M', torch_dtype=torch.bfloat16, trust_remote_code=True).to(device)
glm2_tokenizer = AutoTokenizer.from_pretrained('tattabio/gLM2_650M', trust_remote_code=True)

Loading weights: 100%|██████████| 232/232 [00:00<00:00, 2438.52it/s]
[transformers] gLM2Model LOAD REPORT from: tattabio/gLM2_650M
Key                        | Status     |  | 
---------------------------+------------+--+-
lm_head.proj_output.weight | UNEXPECTED |  | 
lm_head.norm.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 211/211 [00:00<00:00, 6707.58it/s]
[transformers] gLM2Model LOAD REPORT from: tattabio/gLM2_150M
Key                        | Status     |  | 
---------------------------+------------+--+-
lm_head.proj_output.weight | UNEXPECTED |  | 
lm_head.norm.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Loading data, getting embeddings, and calculating NN

In [ ]:
# Loading E coli genome data with annotations
data_path = os.path.join(get_project_root(), "data/imodulondb/e_coli_precise1k")
ecoli_df = pd.read_csv(os.path.join(data_path, "data.csv"))
ecoli_df = ecoli_df[ecoli_df["operon_member"] == False]
ecoli_df = ecoli_df.reset_index(drop=True)
ecoli_df = load_regulon_data(ecoli_df, os.path.join(get_project_root(), "data/regulondb/regulondb.json"))
ecoli_df = load_sigmulon_data(ecoli_df, os.path.join(get_project_root(), "data/regulondb/sigmulon.json"))

In [22]:
# Getting embeddings for all standlone/first-operon member gene promoters from each model

promoter_1_embeddings = compute_promoter_embeddings(model_1,ecoli_df,device,reverse_promoter=checkpoint['args'].reverse_promoter,max_promoter_length=checkpoint['args'].max_promoter_length, batch_size=256)        
promoter_1_distances = pairwise_distances(promoter_1_embeddings, metric="cosine")
promoter_1_nearest_neighbours = np.argsort(promoter_1_distances, axis=1)  

promoter_5_embeddings = compute_promoter_embeddings(model_5,ecoli_df,device,reverse_promoter=checkpoint['args'].reverse_promoter,max_promoter_length=checkpoint['args'].max_promoter_length, batch_size=256)        
promoter_5_distances = pairwise_distances(promoter_5_embeddings, metric="cosine")
promoter_5_nearest_neighbours = np.argsort(promoter_5_distances, axis=1)  

promoter_25_embeddings = compute_promoter_embeddings(model_25,ecoli_df,device,reverse_promoter=checkpoint['args'].reverse_promoter,max_promoter_length=checkpoint['args'].max_promoter_length, batch_size=256)        
promoter_25_distances = pairwise_distances(promoter_25_embeddings, metric="cosine")
promoter_25_nearest_neighbours = np.argsort(promoter_25_distances, axis=1)  

promoter_100_embeddings = compute_promoter_embeddings(model_100,ecoli_df,device,reverse_promoter=checkpoint['args'].reverse_promoter,max_promoter_length=checkpoint['args'].max_promoter_length, batch_size=256)        
promoter_100_distances = pairwise_distances(promoter_100_embeddings, metric="cosine")
promoter_100_nearest_neighbours = np.argsort(promoter_100_distances, axis=1)  



promoter_strings = ecoli_df["promoter"].to_list()
promoter_strings = [promoter[-160:] if len(promoter) > 160 else promoter for promoter in promoter_strings]  
embed_list = []
for promoter in tqdm.tqdm(promoter_strings):
    yep = llm_tokenizer(f"<|bos|>5{promoter}3<|eos|>", return_tensors='pt')
    with torch.no_grad():
        embed_list.append(llm_model.transformer(yep.input_ids.to(device), output_hidden_states=True)['hidden_states'][-2][0].mean(dim=0).cpu().numpy())
llm_promoter_embeddings= np.stack(embed_list, axis=0)        
llm_promoter_distances = pairwise_distances(llm_promoter_embeddings, metric="cosine")
llm_promoter_nearest_neighbours = np.argsort(llm_promoter_distances, axis=1)

promoter_strings = ecoli_df["promoter"].to_list()
promoter_strings = [promoter[-160:] if len(promoter) > 160 else promoter for promoter in promoter_strings]  
embed_list = []
for promoter in tqdm.tqdm(promoter_strings):
    yep = llm_tokenizer(f"<|bos|>5{promoter}3<|eos|>", return_tensors='pt')
    with torch.no_grad():
        embed_list.append(llm_model_small.transformer(yep.input_ids.to(device), output_hidden_states=True)['hidden_states'][-2][0].mean(dim=0).cpu().numpy())
llm_promoter_embeddings_small= np.stack(embed_list, axis=0)        
llm_promoter_distances = pairwise_distances(llm_promoter_embeddings_small, metric="cosine")
llm_promoter_nearest_neighbours_small = np.argsort(llm_promoter_distances, axis=1)

promoter_strings = ecoli_df["promoter"].to_list()
promoter_strings = [promoter[-160:] if len(promoter) > 160 else promoter for promoter in promoter_strings]  
embed_list = []
for promoter in tqdm.tqdm(promoter_strings):
    yep = llm_tokenizer(f"<|bos|>5{promoter}3<|eos|>", return_tensors='pt')
    with torch.no_grad():
        embed_list.append(llm_model_xsmall.transformer(yep.input_ids.to(device), output_hidden_states=True)['hidden_states'][-2][0].mean(dim=0).cpu().numpy())
llm_promoter_embeddings_xsmall= np.stack(embed_list, axis=0)        
llm_promoter_distances = pairwise_distances(llm_promoter_embeddings_xsmall, metric="cosine")
llm_promoter_nearest_neighbours_xsmall = np.argsort(llm_promoter_distances, axis=1)

promoter_strings = ecoli_df["promoter"].to_list()
promoter_strings = [promoter[-300:] if len(promoter) > 300 else promoter for promoter in promoter_strings]  
embed_list = []
for promoter in tqdm.tqdm(promoter_strings):
    yep = glm2_tokenizer(f"<+>{str.lower(promoter)}<+>", return_tensors='pt')
    with torch.no_grad():
        embed_list.append(glm2_650_model(yep.input_ids.to(device), output_hidden_states=True)['hidden_states'][-2][0].mean(dim=0).cpu())
stacked = torch.stack(embed_list).float().numpy()
glm2_promoter_embeddings_650= np.stack(stacked, axis=0)        
glm2_promoter_distances_650 = pairwise_distances(glm2_promoter_embeddings_650, metric="cosine")
glm2_promoter_nearest_neighbours_650= np.argsort(glm2_promoter_distances_650, axis=1) 

promoter_strings = ecoli_df["promoter"].to_list()
promoter_strings = [promoter[-300:] if len(promoter) > 300 else promoter for promoter in promoter_strings]  
embed_list = []
for promoter in tqdm.tqdm(promoter_strings):
    yep = glm2_tokenizer(f"<+>{str.lower(promoter)}<+>", return_tensors='pt')
    with torch.no_grad():
        embed_list.append(glm2_150_model(yep.input_ids.to(device), output_hidden_states=True)['hidden_states'][-2][0].mean(dim=0).cpu())
stacked = torch.stack(embed_list).float().numpy()
glm2_promoter_embeddings_150= np.stack(stacked, axis=0)        
glm2_promoter_distances_150 = pairwise_distances(glm2_promoter_embeddings_150, metric="cosine")
glm2_promoter_nearest_neighbours_150= np.argsort(glm2_promoter_distances_150, axis=1) 

100%|██████████| 2102/2102 [00:23<00:00, 90.27it/s]


In [ ]:
# Getting ESM2 (150M) protein embeddings
protein_embeddings = compute_protein_embeddings(model_1, ecoli_df, batch_size=256, device=device)
protein_distances = pairwise_distances(protein_embeddings, metric="cosine")
protein_nearest_neighbours= np.argsort(protein_distances, axis=1)

### Loading pre-computed Evo1 and Evo2 nearest neighbours

In [47]:
#Loading Evo2 nn
filename = "evo2_7B_nn.pickle"
nn_path = hf_hub_download(repo_id=repo_id, filename=filename )
with open(nn_path, 'rb') as handle:
    evo2_nearest_neighbours = pickle.load(handle) 

In [48]:
#Loading Evo1 nn
filename = "evo1_7B_nn.pickle"
nn_path = hf_hub_download(repo_id=repo_id, filename=filename )
with open(nn_path, 'rb') as handle:
    evo1_nearest_neighbours = pickle.load(handle) 

### Performing evaluation

In [49]:
k = 1
min_members=5

neighbours_sources = [
    promoter_100_nearest_neighbours,     
    promoter_25_nearest_neighbours,     
    promoter_5_nearest_neighbours,
    promoter_1_nearest_neighbours,
    evo2_nearest_neighbours['e_coli_precise1k'], 
    evo1_nearest_neighbours['e_coli_precise1k'],         
    glm2_promoter_nearest_neighbours_650,
    glm2_promoter_nearest_neighbours_150,    
    llm_promoter_nearest_neighbours,   
    llm_promoter_nearest_neighbours_small,   
    llm_promoter_nearest_neighbours_xsmall,   
    protein_nearest_neighbours,
]
source_names = [
    'C3P-large (100M)',
    'C3P-base (25M)',
    'C3P-small (5M)',
    'C3P-tiny (1M)',
    'Evo2 (7B)',
    'Evo1 (7B)',
    'gLM2 (650M)', # 2nd last layer mean
    'gLM2 (150M)',
    'PromoGen2 (149M)', #2nd last layer mean
    'PromoGen2 (33M)',
    'PromoGen2 (8M)',
    'ESM2 (150M)',
]
result_dfs = []

for col in ['regulons', 'sigmulons']:
    results_list = []
    for name, nn_matrix in zip(source_names, neighbours_sources):    
        eval_result = evaluate_knn(
            df=ecoli_df, 
            column_label=col, 
            nearest_neighbours=nn_matrix,
            min_members=min_members,       
            k=k,
        )    
        eval_result_map = evaluate_map(
            ecoli_df,
            col,
            nn_matrix,
            min_members=min_members,
            at_r=True
        )
        
        class_results_map  = eval_result_map.get('class_results', {}) 
        n_classes = len(class_results_map)  
        maps = [res['average_precision'] for res in class_results_map.values()]
        se_maps = 2*(np.std(maps, ddof=1) / np.sqrt(n_classes))
        
        
        class_results = eval_result.get('class_results', {})
        n_classes = len(class_results)    
        print(n_classes)        
        precisions = [res['precision'] for res in class_results.values()]
        recalls = [res['recall'] for res in class_results.values()]
        mccs = [res['mcc'] for res in class_results.values()]
        se_precision = 2*(np.std(precisions, ddof=1) / np.sqrt(n_classes))
        se_recalls = 2*(np.std(recalls, ddof=1) / np.sqrt(n_classes))
        se_mcc = 2*(np.std(mccs, ddof=1) / np.sqrt(n_classes)    )
        results_list.append({
            'Method': name,                             
            'Mean Precision': eval_result.get('mean_precision', np.nan),
            'SE Precision': se_precision,
            'Mean Recall': eval_result.get('mean_recall', np.nan),
            'SE Recall': se_recalls,            
            'Mean MCC': eval_result.get('mean_mcc', np.nan),
            'SE MCC': se_mcc,
            'Mean mAP': eval_result_map.get('mean_average_precision', np.nan),
            'SE mAP': se_maps,
        })        
    result_dfs.append(pd.DataFrame(results_list))

102
102
102
102
102
102
102
102
102
102
102
102
6
6
6
6
6
6
6
6
6
6
6
6


In [50]:
# Regulon prediction results
result_dfs[0][['Method','Mean Precision','SE Precision', 'Mean MCC','SE MCC', 'Mean mAP','SE mAP']]

,Method,Mean Precision,SE Precision,Mean MCC,SE MCC,Mean mAP,SE mAP
0,C3P-large (100M),0.211142,0.042051,0.183901,0.038581,0.057616,0.017352
1,C3P-base (25M),0.180618,0.040347,0.153666,0.035763,0.049801,0.014442
2,C3P-small (5M),0.156062,0.034091,0.137867,0.031853,0.041612,0.012648
3,C3P-tiny (1M),0.127944,0.030519,0.108857,0.027847,0.030787,0.009948
4,Evo2 (7B),0.037325,0.017130,0.018725,0.011433,0.006337,0.001878
5,Evo1 (7B),0.011438,0.006278,-0.002928,0.002861,0.002784,0.001213
6,gLM2 (650M),0.049649,0.017913,0.036098,0.016640,0.009286,0.003779
7,gLM2 (150M),0.030136,0.011874,0.015731,0.010093,0.006267,0.002695
8,PromoGen2 (149M),0.026151,0.015519,0.009318,0.010445,0.003437,0.001611
9,PromoGen2 (33M),0.032171,0.013674,0.017437,0.011945,0.005476,0.002690


In [51]:
# Sigmulon prediction results
result_dfs[1][['Method','Mean Precision','SE Precision', 'Mean MCC','SE MCC', 'Mean mAP','SE mAP']]

,Method,Mean Precision,SE Precision,Mean MCC,SE MCC,Mean mAP,SE mAP
0,C3P-large (100M),0.378694,0.116029,0.295732,0.130209,0.077391,0.051329
1,C3P-base (25M),0.369968,0.116231,0.296191,0.120025,0.080872,0.056049
2,C3P-small (5M),0.354570,0.145204,0.283044,0.157539,0.092878,0.063616
3,C3P-tiny (1M),0.376254,0.125703,0.293493,0.151728,0.108308,0.078755
4,Evo2 (7B),0.157822,0.118296,0.073133,0.055089,0.027884,0.037975
5,Evo1 (7B),0.075221,0.102274,-0.009001,0.014427,0.020640,0.035084
6,gLM2 (650M),0.143742,0.131219,0.062282,0.100598,0.030484,0.039891
7,gLM2 (150M),0.093486,0.119979,0.014896,0.026273,0.022887,0.038110
8,PromoGen2 (149M),0.102156,0.108698,0.020870,0.019691,0.021268,0.035007
9,PromoGen2 (33M),0.099184,0.102037,0.016967,0.012803,0.021018,0.035027
